# Notebook 00 — Setup and data loading

**Purpose.** Verify your environment, load the Synapse deposit, and print sanity-check shapes. No analysis output is produced here — run this notebook first to confirm everything is wired up before working through Notebooks 01–03.

## Required Synapse files

Download version 2 of Synapse dataset `syn75275226` (AD Knowledge Portal). Place all files in a
single directory and set `DEPOSIT_DIR` below. Component IDs are 1–519 and 521–548
(ID 520 is retired), so select components by ID, not by row number.

| File | Shape | Description |
|---|---|---|
| `samples_final.csv` | 547 × 276 | Component × sample loadings |
| `cell_type.csv` | 547 × 6 | Component × cell-type loadings (Ast, Oli, Inh, OPC, Exc, Mic) |
| `RNA.txt` | 25,098 × 547 | Gene × component loadings (PIP > 0.5 masked) |
| `meth.txt` | 51,798 × 547 | CpG × component loadings (PIP > 0.5 masked) |
| `chip.txt` | 26,384 × 547 | H3K9ac peak × component loadings (PIP > 0.5 masked) |

## Required reference annotations

Not bundled with this repository: download the three tables into `references/` (sources and steps in `references/README.md`). The gene annotation is extracted from the Ensembl release 100 GTF; the array manifest and the peak coordinates come with the ROSMAP DNA methylation and H3K9ac ChIP-seq data on the AD Knowledge Portal. All four notebooks load them with the deposit.

| File | Description |
|---|---|
| `Homo_sapiens.GRCh38.100.txt` | Gene annotation (Ensembl 100) |
| `ROSMAP_arrayMethylation_metaData.tsv` | Illumina methylation array manifest |
| `H3K9acDomains.csv` | H3K9ac ChIP-seq peak coordinates (Klein et al.) |

Clinical metadata (`ROSMAP_clinical.csv`) is **not redistributed** — it is
controlled-access (request via the
[AD Knowledge Portal DUC](https://adknowledgeportal.synapse.org/Data%20Access)). Notebooks that need
clinical variables document their dependency inline and degrade gracefully
when the file is absent.


## Step 1 — Imports and helper module

All notebooks share a small helper module at `g_ros_release.py` (at the repo root) that wraps deposit-file loading and the manuscript's plotting / analysis conventions. We import it here as `gr`.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the repo root to sys.path so we can import the helper module.
sys.path.insert(0, "..")
import g_ros_release as gr

gr.apply_plot_style()


## Step 2 — Configure paths and load the deposit

In [2]:
# Path configuration — edit these two paths to match your local layout, or set
# the environment variables G_ROS_DEPOSIT_DIR and G_ROS_REFERENCES_DIR.
# `DEPOSIT_DIR` is where you downloaded the Synapse files (dataset
# syn75275226, version 2). `REFERENCES_DIR` is the `references/`
# directory in this repo, where you put the reference tables.

import os as _os
DEPOSIT_DIR = _os.environ.get("G_ROS_DEPOSIT_DIR", "../synapse")   # <-- your local Synapse download
REFERENCES_DIR = _os.environ.get("G_ROS_REFERENCES_DIR", "../references")   # see references/README.md

deposit = gr.load_deposit(DEPOSIT_DIR, REFERENCES_DIR, load_references=True)
print(f"Loaded {len(deposit.samples)} components × {deposit.samples.shape[1]} samples")
print(f"Cell-type matrix shape: {deposit.cell_type.shape}")
for omic, df in deposit.genes.items():
    print(f"  {omic}: {df.shape[0]} features × {df.shape[1]} components "
          f"(non-NaN cells: {int(df.notna().sum().sum()):,})")


Loaded 547 components × 276 samples
Cell-type matrix shape: (547, 6)
  RNA: 25098 features × 547 components (non-NaN cells: 281,605)
  meth: 51798 features × 547 components (non-NaN cells: 95,582)
  chip: 26384 features × 547 components (non-NaN cells: 69,918)


## Step 3 — Sanity checks

In [3]:
# Cell-type column order in cell_type.csv (NOT the figure plotting order)
print('cell_type.csv columns:', list(deposit.cell_type.columns))
print('Figure plotting order:', gr.CELL_TYPE_PLOT_ORDER)

# Component ID range
print(f'Components: {deposit.samples.index.min()} .. {deposit.samples.index.max()}')
print(f'Total components: {len(deposit.samples)}')


cell_type.csv columns: ['Ast', 'Oli', 'Inh', 'OPC', 'Exc', 'Mic']
Figure plotting order: ['Ast', 'Exc', 'Inh', 'Mic', 'OPC', 'Oli']
Components: 1 .. 548
Total components: 547


## Step 4 — Spot-check the headline components

These are the components featured in the published manuscript. Use `gr.HEADLINE_COMPONENTS` for the full map of component → figure label.

In [4]:
for cid, label in gr.HEADLINE_COMPONENTS.items():
    s = gr.component_summary(deposit, cid)
    print(f'Component {cid:>3} — {label}')
    print(f'    PIP-significant features: '
          f"RNA={s['n_features_RNA']}, meth={s['n_features_meth']}, "
          f"chip={s['n_features_chip']}")
    print(f'    Top cell-type drivers: +{s["top_celltype_pos"] or "none"}, -{s["top_celltype_neg"] or "none"}')


Component  74 — Figure 2: sex-associated regulatory variation
    PIP-significant features: RNA=161, meth=3435, chip=676
    Top cell-type drivers: +Oli, -none
Component 278 — Figure 3: sex-biased component shared by OPCs and mature oligodendrocytes
    PIP-significant features: RNA=1114, meth=0, chip=0
    Top cell-type drivers: +none, -OPC
Component 172 — Figure 4: neuron-dominated component associated with neuritic plaque burden
    PIP-significant features: RNA=1013, meth=0, chip=0
    Top cell-type drivers: +none, -Inh
Component 277 — Figure 5: neuron-dominated component associated with clinical Alzheimer's dementia
    PIP-significant features: RNA=1180, meth=0, chip=0
    Top cell-type drivers: +none, -Exc
Component  21 — Supplementary Figure S7: ectopic olfactory receptor expression and its cis-genetic regulation
    PIP-significant features: RNA=555, meth=0, chip=0
    Top cell-type drivers: +none, -Exc
Component 351 — Supplementary Figure S8: a single-individual multi-omic pr

## Next steps — three analysis-type notebooks

Each downstream notebook operates on a configurable list of components (default: the nine components above, shown in the manuscript figures). Edit the `COMPONENTS = [...]` line near the top of each notebook to focus on a specific component or to sweep all 547.

- **Notebook 01 — Per-component panels.** Sample scatter, cell-type bar, and per-omic chromosome-scatter loadings (RNA / meth / chip). The visual building blocks of every figure in the paper.
- **Notebook 02 — Pathway enrichment.** KEGG pathway enrichment for RNA and methylation (every gene the array manifest's `RefGene` column lists for each CpG). ChIP pathway enrichment is not included because peak-to-gene mapping was external to the deposit.
- **Notebook 03 — Clinical associations.** Data S1 parity tests: Mann-Whitney U for sex, Mann-Whitney U for CERAD 1/2 vs 3/4, Kruskal-Wallis for cogdx groups 1/2/4, with BH/FDR correction across all components. Requires `ROSMAP_clinical.csv` (controlled access).

**Out of scope** for these notebooks: the GWAS and eQTL analyses, the SEA-AD validation and the other analyses described in the manuscript Methods. These need controlled-access ROSMAP data or external datasets and tools; `scripts/README.md` lists the code for them.